In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
feat = pd.read_csv('..data/feature_table.tsv', sep='\t', index_col=0)
feat.columns = feat.columns.str.replace("_merged", "", regex=False)
cat = pd.read_csv('samples_categories.tsv', sep='\t', header=None, names=['sample', 'category'])
cat["sample"] = cat["sample"].str.replace("_merged", "", regex=False)

In [ ]:
X = feat.T 
y = cat.set_index('sample').loc[X.index, 'category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(
    n_estimators=700,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))
print("train accuracy:", rf.score(X_train, y_train))
print("test accuracy:", rf.score(X_test, y_test))

In [ ]:
mdi = pd.Series(rf.feature_importances_, index=X_train.columns)

Extracting important features based on contigues

In [ ]:
from Bio import SeqIO
import pandas as pd

def get_lengths(fasta_path, prefix):
    lengths = {}

    for record in SeqIO.parse(fasta_path, "fasta"):
        feature_num = record.id.split("_")[0]
        feature_name = f"{prefix}_{feature_num}"
        lengths[feature_name] = max(lengths.get(feature_name, 0), len(record.seq))

    return lengths

health_lengths = get_lengths("../data/components.seq_he.fasta", "health")
disease_lengths = get_lengths("../data/components.seq.fasta", "disease")

lengths = {**health_lengths, **disease_lengths}

lengths_df = pd.DataFrame.from_dict(
    lengths, orient="index", columns=["length"]
)

imp_df = mdi.reset_index()
imp_df.columns = ["feature", "importance"]

merged = imp_df.merge(
    lengths_df,
    left_on="feature",
    right_index=True,
    how="left"
)

print(merged.head())

In [ ]:
filtered = merged[merged["length"] >= 100]
top_long = filtered.sort_values("importance", ascending=False).head(20)
print(top_long)

Validation on other samples

In [ ]:
fval = pd.read_csv('../data/feature_table_val.tsv', sep='\t', index_col=0)

In [ ]:
X_val = fval.T
y_val = cat.set_index('sample').loc[X_val.index, 'category']

In [ ]:
y_pred_val = rf.predict(X_val)